# Validation-selected task-trained network (Fig. 2, Table 2, Sec. 3.7, Fig. 6)

The task-trained network used for the calibration diagnostics (Fig. 2), the head comparison (Table 2) and the
frozen-feature transfer (Sec. 3.7, Fig. 6) is the one with the highest validation macro-AUC. This notebook
selects it from the networks saved by notebook 01 and repeats those analyses with the code of
`01_task_trained_classifiers.ipynb`, with fixed seeds.

No network is retrained. The notebook loads the three networks saved by Part 1 of notebook 01
(`_results/checkpoints/`), recomputes their validation and test macro-AUC, and checks them against
`_results/summary_seed42.json`. Random seeds are fixed, so a repeated run gives the same numbers on the same machine.

Output, in `_results/` of the DENTEX folder: `validation_selected.json`, the reliability diagrams (Fig. 2) and the train_564 case
images, separately and stacked as (a) and (b) (Fig. 6). Runtime: a few minutes.

## 1. Environment, device and seeds

In [ ]:
import os, sys, glob, json, random, datetime, importlib, importlib.util, subprocess
import numpy as np, torch
# installs only what this kernel lacks, keeping its torch version
_need = {"torchvision": "torchvision", "sklearn": "scikit-learn", "PIL": "pillow", "matplotlib": "matplotlib"}
_missing = [pkg for mod, pkg in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("installing:", *_missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing,
                    "torch==" + torch.__version__.split("+")[0]], check=False)
    importlib.invalidate_caches()
import torchvision, torch.nn as nn
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)

## 2. Paths and saved networks

In [ ]:
DATA_DIR = "/path/to/DENTEX/training_data/quadrant-enumeration-disease"
IMG_DIR    = os.path.join(DATA_DIR, "xrays")
ANN_PATH   = os.path.join(DATA_DIR, "train_quadrant_enumeration_disease.json")
OUTPUT_DIR = os.path.join(DATA_DIR, "_results")
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")
for p in (IMG_DIR, ANN_PATH, CKPT_DIR):
    assert os.path.exists(p), f"not found: {p}"

ZOO = ["densenet121", "efficientnet_v2_s", "resnet50"]
for n in ZOO:
    assert os.path.exists(os.path.join(CKPT_DIR, f"{n}_seed{SEED}.pt")), \
        f"no saved network for {n}: run Part 1 of 01_task_trained_classifiers.ipynb first"

SUMMARY = os.path.join(OUTPUT_DIR, f"summary_seed{SEED}.json")
SUMMARY_JSON = json.load(open(SUMMARY)) if os.path.exists(SUMMARY) else None
OLD = SUMMARY_JSON["direction1_zoo"] if SUMMARY_JSON else None
print("saved networks found | summary file:", "found" if OLD else "not found")

## 3. Tooth-level targets and the seed-42 radiograph-grouped split (as in notebook 01)

In [ ]:
coco = json.load(open(ANN_PATH))
cats3 = sorted(coco["categories_3"], key=lambda c: c["id"])
DX_NAMES = [c["name"] for c in cats3]; NUM_DX = len(DX_NAMES)
id2idx = {c["id"]: i for i, c in enumerate(cats3)}
img_file = {im["id"]: im["file_name"] for im in coco["images"]}

tooth = {}
for a in coco["annotations"]:
    key = (a["image_id"], a.get("category_id_1", -1), a.get("category_id_2", -1))
    if key not in tooth:
        tooth[key] = {"image_id": a["image_id"], "file": img_file[a["image_id"]],
                      "bbox": list(a["bbox"]), "y": np.zeros(NUM_DX, np.float32)}
    tooth[key]["y"][id2idx[a["category_id_3"]]] = 1.0
    bx,by,bw,bh = tooth[key]["bbox"]; ax,ay,aw,ah = a["bbox"]
    x0,y0 = min(bx,ax), min(by,ay); x1,y1 = max(bx+bw,ax+aw), max(by+bh,ay+ah)
    tooth[key]["bbox"] = [x0,y0,x1-x0,y1-y0]
teeth = list(tooth.values())
Y = np.stack([t["y"] for t in teeth])

VAL_FRAC, TEST_FRAC = 0.15, 0.15
def grouped_split(teeth, seed, val_frac, test_frac):
    img_ids = sorted({t["image_id"] for t in teeth})
    rng = np.random.RandomState(seed); rng.shuffle(img_ids)
    n = len(img_ids); n_te = int(n*test_frac); n_va = int(n*val_frac)
    test_imgs = set(img_ids[:n_te]); val_imgs = set(img_ids[n_te:n_te+n_va])
    tr, va, te = [], [], []
    for i, t in enumerate(teeth):
        if   t["image_id"] in test_imgs: te.append(i)
        elif t["image_id"] in val_imgs:  va.append(i)
        else:                            tr.append(i)
    return tr, va, te
train_idx, val_idx, test_idx = grouped_split(teeth, SEED, VAL_FRAC, TEST_FRAC)
print(len(teeth), "teeth | split", len(train_idx), len(val_idx), len(test_idx))
if SUMMARY_JSON:
    assert [len(train_idx), len(val_idx), len(test_idx)] == [SUMMARY_JSON["split"][k] for k in ("train", "val", "test")], \
        "split differs from summary_seed42.json"

## 4. Tooth crops, metrics and loss weights (as in notebook 01)

In [ ]:
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

CROP, PAD = 224, 0.15
NORM = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
train_tf = T.Compose([T.Grayscale(3), T.Resize((CROP,CROP)), T.RandomHorizontalFlip(0.5),
                      T.ColorJitter(brightness=0.1, contrast=0.1), T.ToTensor(), NORM])
eval_tf  = T.Compose([T.Grayscale(3), T.Resize((CROP,CROP)), T.ToTensor(), NORM])

_img_cache = {}
def load_image(fname):
    if fname not in _img_cache:
        _img_cache[fname] = Image.open(os.path.join(IMG_DIR, os.path.basename(fname))).convert("L")
    return _img_cache[fname]
def crop_tooth(img, bbox, pad=PAD):
    W,H = img.size; x,y,w,h = bbox; px,py = w*pad, h*pad
    x0=max(0,int(x-px)); y0=max(0,int(y-py)); x1=min(W,int(x+w+px)); y1=min(H,int(y+h+py))
    if x1<=x0 or y1<=y0: return img
    return img.crop((x0,y0,x1,y1))
class ToothDS(Dataset):
    def __init__(self, idxs, tf): self.idxs=list(idxs); self.tf=tf; self.c={}
    def __len__(self): return len(self.idxs)
    def __getitem__(self, k):
        i = self.idxs[k]; t = teeth[i]
        if i not in self.c: self.c[i] = crop_tooth(load_image(t["file"]), t["bbox"])
        return self.tf(self.c[i]), torch.from_numpy(t["y"])
BATCH = 32
train_dl = DataLoader(ToothDS(train_idx, train_tf), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(ToothDS(val_idx,   eval_tf), batch_size=BATCH)
test_dl  = DataLoader(ToothDS(test_idx,  eval_tf), batch_size=BATCH)

@torch.no_grad()
def predict(model, dl):
    model.eval(); P=[]; Tt=[]
    for x,y in dl:
        P.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()); Tt.append(y.numpy())
    return np.concatenate(P), np.concatenate(Tt)
def per_condition_auc(yt, yp):
    return {DX_NAMES[c]: (roc_auc_score(yt[:,c], yp[:,c]) if len(np.unique(yt[:,c]))==2 else float("nan"))
            for c in range(NUM_DX)}
def macro_auc(yt, yp):
    v=[a for a in per_condition_auc(yt,yp).values() if not np.isnan(a)]
    return float(np.mean(v)) if v else float("nan")
def ece(yt, yp, n_bins=10):
    bins=np.linspace(0,1,n_bins+1); e=0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(yp>lo)&(yp<=hi)
        if m.sum()==0: continue
        e += (m.sum()/len(yp))*abs(yp[m].mean()-yt[m].mean())
    return float(e)

_pos = Y[train_idx].sum(0); _neg = len(train_idx) - _pos
POS_WEIGHT = torch.tensor(np.clip(_neg/np.maximum(_pos,1), 1.0, 20.0), dtype=torch.float32, device=DEVICE)

## 5. Load the saved networks and select by validation macro-AUC

In [ ]:
from torchvision import models

def build_backbone(name, num_classes):
    # the saved checkpoints hold every weight, so no ImageNet download is needed
    if name == "densenet121":
        m = models.densenet121(weights=None); m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == "efficientnet_v2_s":
        m = models.efficientnet_v2_s(weights=None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif name == "resnet50":
        m = models.resnet50(weights=None); m.fc = nn.Linear(m.fc.in_features, num_classes)
    else:
        raise ValueError(name)
    return m
def load_net(name):
    m = build_backbone(name, NUM_DX)
    m.load_state_dict(torch.load(os.path.join(CKPT_DIR, f"{name}_seed{SEED}.pt"), map_location="cpu"))
    return m.to(DEVICE).eval()

SEL = {}
for n in ZOO:
    m = load_net(n)
    pv, yv = predict(m, val_dl); pt, yt = predict(m, test_dl)
    SEL[n] = {"val_macro_auc": macro_auc(yv, pv), "test_macro_auc": macro_auc(yt, pt)}
    if OLD:
        SEL[n]["val_in_summary"] = OLD[n]["best_val_macro_auc"]; SEL[n]["test_in_summary"] = OLD[n]["test_macro_auc"]
    del m
print(f"{'network':20s} {'val now':>8s} {'val saved':>10s} {'test now':>9s} {'test saved':>11s}")
for n, r in SEL.items():
    print(f"{n:20s} {r['val_macro_auc']:8.4f} {r.get('val_in_summary', float('nan')):10.4f} "
          f"{r['test_macro_auc']:9.4f} {r.get('test_in_summary', float('nan')):11.4f}")
best_name = max(SEL, key=lambda n: SEL[n]["val_macro_auc"])
print("\nSelected by validation macro-AUC:", best_name)
if OLD:
    assert all(abs(SEL[n]["val_macro_auc"] - SEL[n]["val_in_summary"]) < 2e-3 for n in ZOO), \
        "validation AUCs differ from summary_seed42.json: the saved networks are not the ones behind Table 1"

## 6. Reliability diagrams of the selected network (Fig. 2)

In [ ]:
import matplotlib.pyplot as plt
best_model = load_net(best_name)
yp, yt = predict(best_model, test_dl)
FIG2 = {"network": best_name, "test_macro_auc": macro_auc(yt, yp),
        "test_per_condition_auc": per_condition_auc(yt, yp),
        "test_ece": {DX_NAMES[c]: ece(yt[:,c], yp[:,c]) for c in range(NUM_DX)}}
fig, axes = plt.subplots(1, NUM_DX, figsize=(4*NUM_DX, 3.6))
for c in range(NUM_DX):
    p=yp[:,c]; t=yt[:,c]; bins=np.linspace(0,1,11); xs=[]; ys=[]
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        xs.append(p[m].mean()); ys.append(t[m].mean())
    ax=axes[c]; ax.plot([0,1],[0,1],"--",color="gray",lw=1)
    ax.plot(xs,ys,"o-",color="#1a3a5c")
    ax.set_title(f"{DX_NAMES[c]}\nECE={ece(t,p):.3f}", fontsize=10)
    ax.set_xlabel("predicted"); ax.set_ylabel("empirical"); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
FIG2_PATH = os.path.join(OUTPUT_DIR, f"d1_reliability_{best_name}_seed{SEED}_valselected.png")
plt.savefig(FIG2_PATH, dpi=130); plt.show()
print({k: round(v, 4) for k, v in FIG2["test_ece"].items()})

## 7. Frozen features of the selected network and the three heads (Table 2)

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:              # progress bars only
    def tqdm(x, **k): return x
def make_feature_extractor(name):
    m = load_net(name).cpu()
    if name=="densenet121":
        feat = nn.Sequential(m.features, nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        fdim = m.classifier.in_features
    elif name=="efficientnet_v2_s":
        feat = nn.Sequential(m.features, m.avgpool, nn.Flatten()); fdim = m.classifier[1].in_features
    else:
        feat = nn.Sequential(*list(m.children())[:-1], nn.Flatten()); fdim = m.fc.in_features
    return feat.to(DEVICE).eval(), fdim
feat_ext, FDIM = make_feature_extractor(best_name)
print("feature dim:", FDIM)

@torch.no_grad()
def extract(dl):
    F=[]; Y_=[]
    for x,y in tqdm(dl, desc="extract", leave=False):
        F.append(feat_ext(x.to(DEVICE)).cpu()); Y_.append(y)
    return torch.cat(F), torch.cat(Y_)
set_seed(SEED)
Ftr, Ytr = extract(train_dl)      # training crops with the notebook-01 augmentation, as in the original analysis
Fva, Yva = extract(val_dl)
Fte, Yte = extract(test_dl)
print("features:", tuple(Ftr.shape), tuple(Fte.shape))

class PlainHead(nn.Module):
    def __init__(self, f, k): super().__init__(); self.l=nn.Linear(f,k)
    def forward(self,x): return self.l(x)
class IndepHead(nn.Module):
    def __init__(self, f, k):
        super().__init__()
        self.heads=nn.ModuleList([nn.Sequential(nn.Linear(f,64),nn.ReLU(),nn.Linear(64,1)) for _ in range(k)])
    def forward(self,x): return torch.cat([h(x) for h in self.heads],1)
class GatedMLPHead(nn.Module):
    def __init__(self, f, k, hidden=128):
        super().__init__()
        self.shared=nn.Sequential(nn.Linear(f,hidden),nn.ReLU(),nn.Dropout(0.2))
        self.gate  =nn.ModuleList([nn.Linear(hidden,1) for _ in range(k)])
        self.expert=nn.ModuleList([nn.Linear(hidden,1) for _ in range(k)])
    def forward(self,x):
        h=self.shared(x)
        return torch.cat([torch.sigmoid(g(h))*e(h) for g,e in zip(self.gate,self.expert)],1)

from torch.utils.data import TensorDataset
ftr_dl = DataLoader(TensorDataset(Ftr,Ytr), batch_size=64, shuffle=True)
def train_head(head, epochs=40, lr=1e-3):
    head=head.to(DEVICE)
    crit=nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
    opt=torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)
    best,best_state=-1.0,None
    for ep in range(epochs):
        head.train()
        for f,y in ftr_dl:
            f,y=f.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(); loss=crit(head(f),y); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad(): yp=torch.sigmoid(head(Fva.to(DEVICE))).cpu().numpy()
        va=macro_auc(Yva.numpy(),yp)
        if va>best: best,best_state=va,{k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
    head.load_state_dict(best_state)
    return head,best

TABLE2 = {}
set_seed(SEED)
for name, h in {"plain": PlainHead(FDIM,NUM_DX), "independent": IndepHead(FDIM,NUM_DX),
                "gated_mlp": GatedMLPHead(FDIM,NUM_DX)}.items():
    hh, bv = train_head(h)
    with torch.no_grad(): yp = torch.sigmoid(hh(Fte.to(DEVICE))).cpu().numpy()
    yt = Yte.numpy()
    TABLE2[name] = {"val_macro_auc": bv, "test_macro_auc": macro_auc(yt, yp),
                    "test_per_condition_auc": per_condition_auc(yt, yp)}
    print(f"{name:>12}: val={bv:.4f}  test={TABLE2[name]['test_macro_auc']:.4f}  "
          + "  ".join(f"{k}={v:.3f}" for k, v in TABLE2[name]['test_per_condition_auc'].items()))

## 8. Bounded update on the frozen features (Sec. 3.7)

In [ ]:
from sklearn.linear_model import LogisticRegression
DELTA = 1.5
Xtr, Xva, Xte = Ftr.numpy(), Fva.numpy(), Fte.numpy()
ytr, yva, yte = Ytr.numpy(), Yva.numpy(), Yte.numpy()

def fit_baseline(X, Yb):
    return [LogisticRegression(max_iter=1000, C=1.0).fit(X, Yb[:,c]) if len(np.unique(Yb[:,c]))>1 else None
            for c in range(NUM_DX)]
def blogits(probes, X):
    out=np.zeros((len(X),NUM_DX),np.float32)
    for c,p in enumerate(probes): out[:,c]= p.decision_function(X) if p is not None else 0.0
    return out
probes = fit_baseline(Xtr, ytr)
Tp=nn.Parameter(torch.ones(1)); z=torch.tensor(blogits(probes,Xva)); yt_=torch.tensor(yva)
opt=torch.optim.LBFGS([Tp],lr=0.1,max_iter=60); bce=nn.BCEWithLogitsLoss()
opt.step(lambda:(opt.zero_grad(), bce(z/Tp.clamp(min=0.05),yt_).backward(), bce(z/Tp.clamp(min=0.05),yt_))[-1])
TEMP=float(Tp.detach().clamp(min=0.05))

class EnvelopeHead(nn.Module):
    def __init__(s,d,k,delta): super().__init__(); s.delta=delta; s.net=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,k))
    def forward(s,x,base): return base + torch.clamp(s.net(x),-s.delta,s.delta)
set_seed(SEED)
head=EnvelopeHead(Xtr.shape[1],NUM_DX,DELTA).to(DEVICE)
btr=torch.tensor(blogits(probes,Xtr)/TEMP); bva=torch.tensor(blogits(probes,Xva)/TEMP)
dl=DataLoader(TensorDataset(torch.tensor(Xtr),btr,torch.tensor(ytr)),batch_size=64,shuffle=True)
op=torch.optim.AdamW(head.parameters(),lr=1e-3,weight_decay=1e-4); crit=nn.BCEWithLogitsLoss()
best=1e9; bs=None
for ep in range(60):
    head.train()
    for x,b,y in dl:
        x,b,y=x.to(DEVICE),b.to(DEVICE),y.to(DEVICE); op.zero_grad(); crit(head(x,b),y).backward(); op.step()
    head.eval()
    with torch.no_grad(): vl=crit(head(torch.tensor(Xva).to(DEVICE),bva.to(DEVICE)),torch.tensor(yva).to(DEVICE)).item()
    if vl<best: best=vl; bs={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
head.load_state_dict(bs); head.eval()

bte=blogits(probes,Xte)/TEMP
p_base=1/(1+np.exp(-bte))
with torch.no_grad():
    final=head(torch.tensor(Xte).to(DEVICE),torch.tensor(bte).to(DEVICE)).cpu().numpy()
p_env=1/(1+np.exp(-final))
SEC37 = {"network": best_name, "feature_dim": int(FDIM), "temperature": TEMP,
         "base_macro_auc": macro_auc(yte,p_base), "env_macro_auc": macro_auc(yte,p_env),
         "base_macro_ece": float(np.mean([ece(yte[:,c],p_base[:,c]) for c in range(NUM_DX)])),
         "env_macro_ece": float(np.mean([ece(yte[:,c],p_env[:,c]) for c in range(NUM_DX)])),
         "max_logit_dev": float(np.max(np.abs(final-bte)))}
print(f"{best_name} | T={TEMP:.3f} | AUC {SEC37['base_macro_auc']:.4f} -> {SEC37['env_macro_auc']:.4f} | "
      f"ECE {SEC37['base_macro_ece']:.4f} -> {SEC37['env_macro_ece']:.4f} | max|dev|={SEC37['max_logit_dev']:.4f}")

## 9. Case figures on train_564 (Fig. 6)

The first image shows the network's own outputs; the second shows the same teeth after the bounded update, with
teeth beyond the 95th-percentile validation distance marked as deferred. Drawing code as in notebook 01.

In [ ]:
from PIL import ImageDraw, ImageFont
TARGET = "train_564"
rows = [(i,t) for i,t in enumerate(teeth) if TARGET in os.path.basename(t["file"])]
assert rows, f"{TARGET} not found"
COLORS = {0:(170,130,255),1:(240,200,40),2:(80,160,255),3:(255,70,70)}
def new_canvas():
    g = load_image(teeth[rows[0][0]]["file"]); img = g.convert("RGB"); d = ImageDraw.Draw(img)
    try: f = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 26)
    except Exception: f = ImageFont.load_default()
    return g, img, d, f
def draw_labels(img, draw, font, placed):
    rects=[]
    def ov(r): return any(not (r[2]<q[0] or r[0]>q[2] or r[3]<q[1] or r[1]>q[3]) for q in rects)
    def tsize(s): b=draw.textbbox((0,0),s,font=font); return b[2]-b[0], b[3]-b[1]
    placed.sort(key=lambda r:(r[1],r[0]))
    for x0,y0,x1,y1,label,c in placed:
        draw.rectangle([x0,y0,x1,y1],outline=c,width=3)
        tw,th=tsize(label); lx=x0; ly=y0-th-8; tries=0
        while (ly<2 or ov((lx,ly,lx+tw+8,ly+th+6))) and tries<12:
            ly-=(th+8)
            if ly<2: ly=y1+6+tries*(th+8)
            tries+=1
        lx=max(2,min(lx,img.width-tw-8))
        draw.rectangle([lx,ly,lx+tw+8,ly+th+6],fill=(0,0,0))
        draw.text((lx+4,ly+2),label,fill=c,font=font)
        rects.append((lx,ly,lx+tw+8,ly+th+6))

# (a) the network's own outputs
g, img, draw, font = new_canvas(); placed=[]; CASE_RAW=[]
with torch.no_grad():
    for i,t in rows:
        p = torch.sigmoid(best_model(eval_tf(crop_tooth(g, t["bbox"])).unsqueeze(0).to(DEVICE)))[0].cpu().numpy()
        x0,y0,w,h = t["bbox"]; top=int(np.argmax(p))
        CASE_RAW.append({"tooth": i, "top": DX_NAMES[top], "prob": float(p[top]), "shown": bool(p[top]>=0.5)})
        if p[top]>=0.5: placed.append([x0,y0,x0+w,y0+h,f"{DX_NAMES[top]} {p[top]*100:.0f}%",COLORS.get(top,(255,255,255))])
        else: draw.rectangle([x0,y0,x0+w,y0+h],outline=(90,200,90),width=2)
draw_labels(img, draw, font, placed)
RAW_PATH = os.path.join(OUTPUT_DIR, f"{TARGET}_{best_name}_RAW.png"); img.save(RAW_PATH)

# (b) reference probe, clipped residual and Mahalanobis gate on the same features
mu = Xtr.mean(0); ci = np.linalg.inv(np.cov(Xtr.T) + np.eye(Xtr.shape[1])*1e-2)
def maha(X): return np.einsum("ni,ij,nj->n", X-mu, ci, X-mu)
ood_thr = float(np.quantile(maha(Xva), 0.95))
g, img, draw, font = new_canvas(); placed=[]; CASE_ENV=[]
with torch.no_grad():
    for i,t in rows:
        emb = feat_ext(eval_tf(crop_tooth(g, t["bbox"])).unsqueeze(0).to(DEVICE)).cpu().numpy()
        d = float(maha(emb)[0]); x0,y0,w,h = t["bbox"]
        if d>ood_thr:
            CASE_ENV.append({"tooth": i, "deferred": True, "distance": d})
            placed.append([x0,y0,x0+w,y0+h,"Deferred (OOD)",(220,220,220)]); continue
        bl = blogits(probes, emb)/TEMP
        fin = head(torch.tensor(emb).to(DEVICE), torch.tensor(bl).to(DEVICE)).cpu().numpy()[0]
        p = 1/(1+np.exp(-fin)); top=int(np.argmax(p))
        CASE_ENV.append({"tooth": i, "deferred": False, "distance": d, "top": DX_NAMES[top], "prob": float(p[top]),
                         "shown": bool(p[top]>=0.5)})
        if p[top]>=0.5: placed.append([x0,y0,x0+w,y0+h,f"{DX_NAMES[top]} {p[top]*100:.0f}%",COLORS.get(top,(255,255,255))])
        else: draw.rectangle([x0,y0,x0+w,y0+h],outline=(90,200,90),width=2)
draw_labels(img, draw, font, placed)
ENV_PATH = os.path.join(OUTPUT_DIR, f"{TARGET}_{best_name}_ENVELOPED.png"); img.save(ENV_PATH)
print(f"{len(rows)} teeth | shown raw {sum(r['shown'] for r in CASE_RAW)} | deferred {sum(r['deferred'] for r in CASE_ENV)} "
      f"| validation threshold {ood_thr:.1f}")

# (a) above (b), as in the manuscript figure
def stack(paths, out):
    ims = [Image.open(p).convert("RGB") for p in paths]
    w = max(i.width for i in ims) + 30; lab_h = 50
    canvas = Image.new("RGB", (w, sum(i.height + lab_h for i in ims) + 20), "white"); d = ImageDraw.Draw(canvas)
    try: f = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 30)
    except Exception: f = ImageFont.load_default()
    y = 10
    for lab, im in zip(["(a)", "(b)"], ims):
        d.text((15, y + 8), lab, fill="black", font=f); y += lab_h
        canvas.paste(im, (15, y)); y += im.height
    canvas.save(out)
FIG6_PATH = os.path.join(OUTPUT_DIR, f"fig6_{TARGET}_{best_name}.png")
stack([RAW_PATH, ENV_PATH], FIG6_PATH)
print("saved:", RAW_PATH, ENV_PATH, FIG6_PATH, sep="\n  ")

## 10. Save

In [ ]:
res = {"version": "validation_selected_v1", "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
       "software": {"torch": torch.__version__, "torchvision": torchvision.__version__, "device": DEVICE},
       "seed": SEED, "split_sizes": [len(train_idx), len(val_idx), len(test_idx)],
       "selection": SEL, "selected_network": best_name, "fig2": FIG2, "table2": TABLE2, "sec37": SEC37,
       "fig6": {"raw": CASE_RAW, "enveloped": CASE_ENV, "ood_threshold": ood_thr},
       "files": [os.path.basename(FIG2_PATH), os.path.basename(RAW_PATH), os.path.basename(ENV_PATH),
                 os.path.basename(FIG6_PATH)]}
path = os.path.join(OUTPUT_DIR, "validation_selected.json")
json.dump(res, open(path, "w"), indent=1, default=float)
print("Saved", path)